In [3]:
import pandas as pd

customers = pd.read_csv("Client data/customers.csv")
products = pd.read_csv("Client data/products.csv")
ratings = pd.read_csv("Client data/ratings.csv")
orders = pd.read_csv("Client data/orders.csv")


# Convert 'order_date' column to order datetime format
orders['order_date'] = pd.to_datetime(orders['order_date'])

In [9]:
# merge datasets

# merge_data = orders.merge(products, on='product_id').merge(customers, on='customer_id')
# merge_data

# Step 2: Prepare the data

#### Convert dates and calculate revenue per order.

In [7]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Merge orders with products to get price information
sales = orders.merge(
    products[['product_id', 'product_name', 'price', 'category']],
    on='product_id',
    how='left'
)

# Revenue per transaction
sales['revenue'] = sales['quantity'] * sales['price']

# Top Performing Products by Revenue

In [8]:
top_products_revenue = (
    sales.groupby(['product_id', 'product_name'])
         .agg(
             total_revenue=('revenue', 'sum'),
             total_units=('quantity', 'sum')
         )
         .sort_values('total_revenue', ascending=False)
         .reset_index()
)

print(top_products_revenue.head(10))

   product_id         product_name  total_revenue  total_units
0           6  Brimnes Bed Storage          59521           77
1          42     Småstad Wardrobe          51688           52
2          29         Råskog Stool          50445           59
3          26        Docksta Table          46386           54
4          31         Nockeby Sofa          45646           58
5           5        Hemnes Daybed          44785           53
6          34   Fjälla Storage Box          43384           44
7          48       Bestå TV Bench          42320           46
8           1          Ektorp Sofa          40946           59
9          46   Valje Wall Cabinet          40140           60


# Top Performing Products by Units Sold

In [10]:
top_products_units = (
    sales.groupby(['product_id', 'product_name'])
         .agg(
             total_units=('quantity', 'sum'),
             total_revenue=('revenue', 'sum')
         )
         .sort_values('total_units', ascending=False)
         .reset_index()
)

print(top_products_units.head(10))

   product_id             product_name  total_units  total_revenue
0          39    Mackapar Shoe Storage           80          36640
1           6      Brimnes Bed Storage           77          59521
2          49   Söderhamn Sofa Section           63          27279
3           2           Poäng Armchair           63          35847
4          38  Bekant Conference Table           62          27342
5           8    Melltorp Dining Table           62          22878
6          30     Strandmon Wing Chair           61           5002
7          46       Valje Wall Cabinet           60          40140
8           1              Ektorp Sofa           59          40946
9          22     Nordli Chest Drawers           59          33099


# Identify Top Clients for the Last Month

#### Determine the most recent month in the data.

In [11]:
latest_date = sales['order_date'].max()

last_month_sales = sales[
    sales['order_date'] >= (latest_date - pd.DateOffset(months=1))
]

#### Join customer information.

In [12]:
last_month_sales = last_month_sales.merge(
    customers,
    on='customer_id',
    how='left'
)

#### Top Clients by Revenue

In [13]:
top_clients_revenue = (
    last_month_sales.groupby(['customer_id', 'name'])
                    .agg(
                        total_revenue=('revenue', 'sum'),
                        total_orders=('order_id', 'count'),
                        total_units=('quantity', 'sum')
                    )
                    .sort_values('total_revenue', ascending=False)
                    .reset_index()
)

print(top_clients_revenue.head(10))

   customer_id         name  total_revenue  total_orders  total_units
0            1   Customer_1          19755            12           29
1           33  Customer_33          17302            10           24
2            4   Customer_4          16562            13           34
3           81  Customer_81          15147             9           25
4           52  Customer_52          14939            11           25
5           50  Customer_50          14807             9           22
6           44  Customer_44          14473             7           20
7           54  Customer_54          14367            10           22
8           18  Customer_18          14356             8           23
9            8   Customer_8          14156             8           25


#### Top Clients by Quantity Purchased

In [14]:
top_clients_units = (
    last_month_sales.groupby(['customer_id', 'name'])
                    .agg(
                        total_units=('quantity', 'sum'),
                        total_revenue=('revenue', 'sum')
                    )
                    .sort_values('total_units', ascending=False)
                    .reset_index()
)

print(top_clients_units.head(10))

   customer_id         name  total_units  total_revenue
0            4   Customer_4           34          16562
1            1   Customer_1           29          19755
2           81  Customer_81           25          15147
3            8   Customer_8           25          14156
4           52  Customer_52           25          14939
5           72  Customer_72           24          14045
6           47  Customer_47           24          14146
7           33  Customer_33           24          17302
8           18  Customer_18           23          14356
9           21  Customer_21           23           8430


#### Bonus: Include Product Ratings

In [15]:
# to see whether highly-rated products are also top sellers

avg_ratings = (
    ratings.groupby('product_id')
           .agg(avg_rating=('rating', 'mean'))
           .reset_index()
)

product_performance = (
    top_products_revenue.merge(
        avg_ratings,
        on='product_id',
        how='left'
    )
)

print(product_performance.head(10))

   product_id         product_name  total_revenue  total_units  avg_rating
0           6  Brimnes Bed Storage          59521           77    3.636364
1          42     Småstad Wardrobe          51688           52    2.714286
2          29         Råskog Stool          50445           59    3.428571
3          26        Docksta Table          46386           54    2.857143
4          31         Nockeby Sofa          45646           58    3.000000
5           5        Hemnes Daybed          44785           53    2.500000
6          34   Fjälla Storage Box          43384           44    2.750000
7          48       Bestå TV Bench          42320           46    3.100000
8           1          Ektorp Sofa          40946           59    2.800000
9          46   Valje Wall Cabinet          40140           60    3.000000


# Executive Summary Table

#### A useful dashboard-style summary:

In [16]:
summary = (
    sales.groupby(['product_id', 'product_name', 'category'])
         .agg(
             total_revenue=('revenue', 'sum'),
             total_units=('quantity', 'sum')
         )
         .reset_index()
)

summary = summary.merge(
    ratings.groupby('product_id')['rating']
           .mean()
           .reset_index()
           .rename(columns={'rating': 'avg_rating'}),
    on='product_id',
    how='left'
)

summary = summary.sort_values(
    'total_revenue',
    ascending=False
)

print(summary.head(20))

    product_id           product_name           category  total_revenue  \
5            6    Brimnes Bed Storage               Beds          59521   
41          42       Småstad Wardrobe  Storage Solutions          51688   
28          29           Råskog Stool             Chairs          50445   
25          26          Docksta Table     Tables & Desks          46386   
30          31           Nockeby Sofa  Sofas & Armchairs          45646   
4            5          Hemnes Daybed               Beds          44785   
33          34     Fjälla Storage Box  Storage Solutions          43384   
47          48         Bestå TV Bench              Decor          42320   
0            1            Ektorp Sofa  Sofas & Armchairs          40946   
45          46     Valje Wall Cabinet  Storage Solutions          40140   
36          37            Fredde Desk     Tables & Desks          39422   
26          27           Ivar Cabinet  Storage Solutions          37785   
9           10   Kallax S